In [ ]:
import os, pathlib, re, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-full-corpus"
if not (pathlib.Path(TAPT_OUT) / "config.json").exists():
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--corpus", "data/raw/binary_train.csv", "data/raw/binary_validation_inputs.csv",
         "data/raw/multiclass_train.csv", "data/raw/multiclass_validation_inputs.csv",
         "data/external/offenseval_kn.csv",
         "--allow-transductive", "--val-frac", "0", "--min-words", "1", "--epochs", "8",
         "--out", TAPT_OUT], log="artifacts/logs/tapt_full_corpus.log")

In [ ]:
SEEDS = ["42", "43", "44"]
MURIL_TAG = "a_final_muril"
run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", MURIL_TAG,
     "--model", TAPT_OUT, "--folds", "1", "--seeds", *SEEDS, "--epochs", "10",
     "--reinit-layers", "2", "--bs", "8", "--grad-accum", "2", "--eval-bs", "32",
     "--select", "last", "--transductive"], log=f"artifacts/logs/{MURIL_TAG}.log")
log = pathlib.Path(f"artifacts/logs/{MURIL_TAG}.log").read_text()
assert re.findall(r"seed (\d+) FULL FIT", log) == SEEDS
assert "transductive:" in log
muril_p = np.load(pathlib.Path("artifacts/runs") / MURIL_TAG / "test_probs.npy")

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from hastika.common.preprocessing import clean, dedupe_index
from hastika.task_a.leak import derive, hidden_test

tr = pd.read_csv("data/raw/binary_train.csv")
tr = tr.iloc[dedupe_index(tr["Comment"].tolist(), tr["Label"].tolist())].reset_index(drop=True)
extra = pd.concat([derive(include_uncertain=True), hidden_test()],
                  ignore_index=True).drop_duplicates("id")
X = [clean(t, demojize=True) for t in list(tr["Comment"]) + list(extra["Comment"])]
y = (pd.concat([tr["Label"], extra["Label"]]) == "Hate").astype(int).values
va = pd.read_csv("data/raw/binary_validation_inputs.csv")
Xv = [clean(t, demojize=True) for t in va["Comment"]]
svm = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    CalibratedClassifierCV(LinearSVC(C=0.5, class_weight="balanced"), cv=3))
svm_p = svm.fit(X, y).predict_proba(Xv)
print(len(y), svm_p.shape, muril_p.shape)
assert svm_p.shape == muril_p.shape

In [ ]:
W_SVM, THRESH = 0.4, 0.5
p = W_SVM * svm_p[:, 1] + (1 - W_SVM) * muril_p[:, 1]
out = pathlib.Path("artifacts/runs/a_final")
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": va["id"], "label": np.where(p > THRESH, "Hate", "Non-Hate")}
             ).to_csv(out / "predictions.csv", index=False)
ZIP = "/kaggle/working/task_a_final.zip"
run([sys.executable, "-m", "hastika.common.submission", "--task", "a",
     "--pred", str(out / "predictions.csv"), "--out", ZIP])
with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["predictions.csv"]
print(pd.read_csv(out / "predictions.csv")["label"].value_counts().to_dict())
run([sys.executable, "-m", "hastika.task_a.leak", "--check", ZIP])

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_final_outputs")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / "task_a_final.zip")
shutil.copy2(out / "predictions.csv", OUT / "predictions.csv")
np.save(OUT / "muril_test_probs.npy", muril_p)
np.save(OUT / "svm_test_probs.npy", svm_p)
for f in pathlib.Path("artifacts/logs").glob("*.log"):
    shutil.copy2(f, OUT / f.name)
print(sorted(x.name for x in OUT.iterdir()))